# Lab 01 - Prompt Anatomy, Contracts, and Validation
### Week 3 - Prompt Engineering and Task-to-Prompt Mapping

You already treat function signatures and API responses as contracts. In this lab you do the same thing to prompts. You will take messy requests, engineer them into structured prompts, define an **output contract as code**, and build a **validator and a linter** that treat prompt output the way unit tests treat function output.

**Why offline and deterministic.** This notebook never calls a live model. A tiny deterministic stand-in (`mock_classify`) plays the role of the model so the whole loop runs cold with zero API keys and identical results every run. The engineering you are graded on here is the prompt structure and the validation around the model, not the model's cleverness. An optional appendix shows how the same contract plugs into a real provider call.

**Stack.** Python 3.13, `pydantic` 2.13.4, `orjson` 3.11.9. No other third-party dependencies.

---

### Learning outcomes
By the end you will be able to:
1. Assemble a prompt from its four parts: instruction, context, constraints, and output contract.
2. Detect zero-shot, one-shot, and few-shot prompts programmatically.
3. Encode an output contract as `pydantic` models with field and cross-field validators.
4. Validate raw model output without ever letting a bad payload crash your pipeline.
5. Lint a prompt for anti-patterns (missing contract, missing abstain, unbounded output, mixed intents).
6. Build few-shot prompts that provably do not leak evaluation items.

### How to work
Each task has a stub that raises `NotImplementedError`. Replace the stub body so the check cell below it turns green. The `check()` helper is soft: a failing or unfinished task prints `FAIL` or `ERROR` but never stops the notebook. Run top to bottom; the final tally tells you where you stand.


## Part A - Setup (given)

Run the next three cells as-is. They provide the synthetic data, the soft check harness, and the deterministic model stand-in. Read `mock_classify` closely: it is the only thing standing in for an LLM, and it emits output in exactly the contract shape you will validate later.

In [ ]:
%pip install -r requirements.txt

In [ ]:
from __future__ import annotations

import re
from dataclasses import dataclass, field

import orjson
from pydantic import BaseModel, ValidationError, field_validator, model_validator

# Synthetic, clearly fictional support tickets for the made-up retailer "Larkfield".
TICKETS = [
    {"ticket_id": "LK-1001",
     "text": "I was charged twice for order 5582 last Friday and need one charge reversed."},
    {"ticket_id": "LK-1002",
     "text": "Checkout throws a 500 error every time I add more than five items to the cart."},
    {"ticket_id": "LK-1003",
     "text": "Please add a dark mode to the mobile app, my eyes hurt when I shop at night."},
    {"ticket_id": "LK-1004",
     "text": "I cannot reset my password, the reset email never arrives even after ten tries."},
    {"ticket_id": "LK-1005",
     "text": "Search takes over thirty seconds to return results every evening around dinner."},
    {"ticket_id": "LK-1006",
     "text": "Hi team, just wanted to say the new storefront logo looks really nice."},
    {"ticket_id": "LK-1007",
     "text": "The orders screen is painfully slow and it also crashes about half the time I open it."},
]

ALLOWED_LABELS = {"billing", "bug", "feature", "account", "performance", "unknown"}
print(f"{len(TICKETS)} tickets loaded. Labels: {sorted(ALLOWED_LABELS)}")

In [ ]:
# Soft check harness. Records results, never raises, so an unfinished task cannot
# stop the notebook. Re-run the whole notebook to reset the tally.
_RESULTS = {"pass": 0, "fail": 0, "error": 0}

def check(name: str, condition, detail: str = "") -> None:
    """Soft assertion. Prints PASS / FAIL / ERROR and tallies. Never raises."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception as exc:
        _RESULTS["error"] += 1
        print(f"ERROR {name}: {type(exc).__name__}: {exc}")
        return
    if ok:
        _RESULTS["pass"] += 1
        print(f"PASS  {name}")
    else:
        _RESULTS["fail"] += 1
        print(f"FAIL  {name}{'  -> ' + detail if detail else ''}")

def reset_results() -> None:
    _RESULTS.update({"pass": 0, "fail": 0, "error": 0})

def summary() -> None:
    print("-" * 52)
    print(f"PASS={_RESULTS['pass']}  FAIL={_RESULTS['fail']}  ERROR={_RESULTS['error']}")

print("check harness ready")

In [ ]:
# Deterministic stand-in for a model call. NOT an LLM. It exists only so the
# prompt -> output -> validate loop runs offline and reproducibly. The teaching
# point of this lab is prompt structure and output-contract validation, not the
# cleverness of this classifier.
_KEYWORDS = [
    ("billing", ("charge", "charged", "refund", "invoice", "payment", "tax")),
    ("performance", ("slow", "seconds", "timeout", "timed out", "lag")),
    ("bug", ("error", "crash", "crashes", "500", "broken", "freeze", "freezes")),
    ("account", ("password", "login", "log in", "sign in", "signup", "profile")),
    ("feature", ("please add", "feature", "would be nice", "wish", "dark mode")),
]

def mock_classify(tickets: list[dict]) -> dict:
    """Return contract-shaped classification JSON for the given tickets."""
    records = []
    for t in tickets:
        low = t["text"].lower()
        hits = [label for label, kws in _KEYWORDS if any(k in low for k in kws)]
        if not hits:
            label = "unknown"
            rationale = "No clear support signal in the message, insufficient evidence to label."
        elif set(hits) >= {"bug", "performance"}:
            label = "bug"  # tie-break: prefer bug over performance when both plausible
            rationale = "Mentions both a crash and slowness, tie-break prefers bug over performance."
        else:
            label = hits[0]
            rationale = f"Message signals a {label} issue based on its wording and intent."
        records.append({"ticket_id": t["ticket_id"], "label": label, "rationale": rationale})
    return {"records": records, "stats": {"count": len(records)}}

demo = mock_classify(TICKETS)
for r in demo["records"]:
    print(f'{r["ticket_id"]} -> {r["label"]}')

## Part B - Task 1: Assemble a prompt from its four parts

A robust prompt has four labeled sections: **instruction**, **context**, **constraints**, and **output contract**. Build the assembler that renders them in that canonical order.

**Contract for `build_prompt`:**
- Return a single string with four sections in this order: `Instruction:`, `Context:`, `Constraints:`, `Output contract:`.
- Each section header sits on its own line, followed by that section's body.
- `constraints` is a list of strings, rendered one per line, each line prefixed with `- ` (dash, space).
- Sections are separated by one blank line. Strip incidental leading and trailing whitespace on the bodies.

In [ ]:
def build_prompt(instruction: str, context: str, constraints: list[str],
                 output_contract: str) -> str:
    parts = [
        "Instruction:\n" + instruction.strip(),
        "Context:\n" + context.strip(),
        "Constraints:\n" + "\n".join(f"- {c.strip()}" for c in constraints),
        "Output contract:\n" + output_contract.strip(),
    ]
    return "\n\n".join(parts)

In [ ]:
_HEADERS = ("Instruction:", "Context:", "Constraints:", "Output contract:")
try:
    _p1 = build_prompt(
        instruction="Classify each ticket into exactly one label.",
        context="Definitions: billing, bug, feature, account, performance, unknown.",
        constraints=["Cite a short rationale.", "Use abstain policy: choose unknown."],
        output_contract='{ "records": [...], "stats": {"count": <int>} }')
except Exception:
    _p1 = None
check("T1 all four headers present", lambda: all(_p1.find(h) != -1 for h in _HEADERS))
check("T1 headers in canonical order",
      lambda: [_p1.find(h) for h in _HEADERS] == sorted(_p1.find(h) for h in _HEADERS))
check("T1 constraints bulleted", lambda: "- Cite a short rationale." in _p1)

## Task 2: Detect the shot type

Zero-shot, one-shot, and few-shot differ only by how many worked examples the prompt carries. Write the detector.

**Contract for `count_shots`:**
- Input is a prompt string. Count the worked examples in it.
- An example is any line whose stripped, lower-cased form starts with the word `example` (so `Example LK-9:` counts).
- Return a tuple `(kind, n)` where `n` is the count and `kind` is `"zero"` for 0, `"one"` for exactly 1, and `"few"` for 2 or more.

In [ ]:
def count_shots(prompt: str) -> tuple[str, int]:
    n = sum(1 for line in prompt.splitlines()
            if line.strip().lower().startswith("example"))
    kind = "zero" if n == 0 else "one" if n == 1 else "few"
    return kind, n

In [ ]:
_zero = "Instruction:\nClassify.\nOutput contract:\n{...}"
_one = _zero + "\n\nExample LK-9:\nTicket: x\nOutput:\n{...}"
_few = _one + "\n\nExample LK-8:\nTicket: y\nOutput:\n{...}"
check("T2 zero-shot", lambda: count_shots(_zero) == ("zero", 0))
check("T2 one-shot", lambda: count_shots(_one) == ("one", 1))
check("T2 few-shot", lambda: count_shots(_few) == ("few", 2))

## Part C - Task 3: Encode the output contract as pydantic models

The output contract is not a comment, it is code that rejects malformed output. Complete three models so the contract is enforced on construction.

**Contract:**
- `Record` has `ticket_id: str`, `label: str`, `rationale: str`.
  - `label` must be a member of `ALLOWED_LABELS`, else raise a validation error.
  - `rationale` must contain between 5 and 40 words inclusive (split on whitespace), else raise.
- `Stats` has `count: int`.
- `ClassificationOutput` has `records: list[Record]` and `stats: Stats`, and after construction must enforce that `stats.count` equals the number of records.

Use `pydantic` v2 idioms: `@field_validator` with `@classmethod`, and `@model_validator(mode="after")` for the cross-field rule.

In [ ]:
class Record(BaseModel):
    ticket_id: str
    label: str
    rationale: str

    @field_validator("label")
    @classmethod
    def label_in_allowed_set(cls, v: str) -> str:
        if v not in ALLOWED_LABELS:
            raise ValueError(f"label {v!r} not in allowed set {sorted(ALLOWED_LABELS)}")
        return v

    @field_validator("rationale")
    @classmethod
    def rationale_word_bounds(cls, v: str) -> str:
        words = len(v.split())
        if not (5 <= words <= 40):
            raise ValueError(f"rationale must be 5 to 40 words, got {words}")
        return v


class Stats(BaseModel):
    count: int


class ClassificationOutput(BaseModel):
    records: list[Record]
    stats: Stats

    @model_validator(mode="after")
    def count_matches_records(self) -> "ClassificationOutput":
        if self.stats.count != len(self.records):
            raise ValueError(
                f"stats.count ({self.stats.count}) != number of records ({len(self.records)})")
        return self

In [ ]:
def _t3_constructs() -> bool:
    ClassificationOutput.model_validate(mock_classify(TICKETS))
    return True

def _t3_raises(payload) -> bool:
    try:
        ClassificationOutput.model_validate(payload); return False
    except ValidationError:
        return True

check("T3 valid output constructs", _t3_constructs)
check("T3 rejects bad label", lambda: _t3_raises(
    {"records": [{"ticket_id": "X", "label": "urgent",
                  "rationale": "this rationale is clearly long enough to pass the bound"}],
     "stats": {"count": 1}}))
check("T3 rejects short rationale", lambda: _t3_raises(
    {"records": [{"ticket_id": "X", "label": "bug", "rationale": "too short"}],
     "stats": {"count": 1}}))
check("T3 rejects count mismatch", lambda: _t3_raises(
    {"records": [{"ticket_id": "X", "label": "bug",
                  "rationale": "a rationale long enough to clear the five word floor"}],
     "stats": {"count": 9}}))

## Task 4: Validate raw output without ever crashing

Real model output arrives as bytes or text and is sometimes malformed. Wrap parsing and validation so a bad payload returns a report instead of an exception.

**Contract for `validate_output`:**
- Input `raw` is a `str` or `bytes` JSON payload. Parse it with `orjson.loads`.
- If parsing fails, return `ValidationReport(ok=False, errors=[...])` describing the parse failure.
- Otherwise validate against `ClassificationOutput`. On success return `ValidationReport(ok=True, errors=[])`.
- On a validation failure, return `ok=False` with one human-readable string per problem. Include the field location and the message.
- The function must never raise. `ValidationReport` is given below.

In [ ]:
@dataclass
class ValidationReport:
    ok: bool
    errors: list[str] = field(default_factory=list)

In [ ]:
def validate_output(raw) -> ValidationReport:
    try:
        data = orjson.loads(raw)
    except orjson.JSONDecodeError as exc:
        return ValidationReport(ok=False, errors=[f"invalid JSON: {exc}"])
    try:
        ClassificationOutput.model_validate(data)
    except ValidationError as exc:
        msgs = [f"{'.'.join(str(p) for p in e['loc'])}: {e['msg']}" for e in exc.errors()]
        return ValidationReport(ok=False, errors=msgs)
    return ValidationReport(ok=True, errors=[])

In [ ]:
_bad_payload = orjson.dumps(
    {"records": [{"ticket_id": "X", "label": "nope",
                  "rationale": "a rationale long enough to clear the floor here"}],
     "stats": {"count": 1}})
check("T4 valid payload ok",
      lambda: validate_output(orjson.dumps(mock_classify(TICKETS))).ok)
check("T4 invalid payload not ok", lambda: not validate_output(_bad_payload).ok)
check("T4 invalid payload reports why", lambda: len(validate_output(_bad_payload).errors) >= 1)
check("T4 malformed JSON does not raise", lambda: not validate_output('{"records": [}').ok)

## Part D - Task 5: Lint a prompt for anti-patterns

A linter for prompts is unit testing for prompt hygiene. Return a list of findings, each a `Finding(code, message)`.

**Detect exactly these four, using case-insensitive matching:**
- `MISSING_CONTRACT` - the text `output contract` does not appear.
- `MISSING_ABSTAIN` - neither `abstain` nor `unknown` appears.
- `UNBOUNDED_OUTPUT` - there is no numeric length constraint AND no output contract. A numeric length constraint is a number immediately followed by one of `word(s)`, `bullet(s)`, `char(s)`, `characters`, `token(s)`.
- `MIXED_INTENTS` - more than one distinct task verb appears, among the stems `classif`, `extract`, `summari`, `generat`.

Return findings **sorted by `code`**. A clean prompt returns an empty list. `Finding` is given below.

In [ ]:
@dataclass
class Finding:
    code: str
    message: str

In [ ]:
_LENGTH_RE = re.compile(
    r"\b\d+\s*(word|words|bullet|bullets|char|characters|token|tokens)\b", re.IGNORECASE)
_VERB_STEMS = {"classif": "classify", "extract": "extract",
               "summari": "summarize", "generat": "generate"}

def lint_prompt(prompt: str) -> list[Finding]:
    low = prompt.lower()
    findings: list[Finding] = []
    if "output contract" not in low:
        findings.append(Finding("MISSING_CONTRACT", "no 'Output contract:' section found"))
    if "abstain" not in low and "unknown" not in low:
        findings.append(Finding("MISSING_ABSTAIN", "no abstain or unknown policy found"))
    if not _LENGTH_RE.search(prompt) and "output contract" not in low:
        findings.append(Finding("UNBOUNDED_OUTPUT",
                                "no length constraint and no output contract to bound output"))
    verbs = {canon for stem, canon in _VERB_STEMS.items() if stem in low}
    if len(verbs) > 1:
        findings.append(Finding("MIXED_INTENTS",
                                f"multiple task verbs present: {', '.join(sorted(verbs))}"))
    return sorted(findings, key=lambda f: f.code)

In [ ]:
try:
    _clean = build_prompt(
        instruction="Classify each ticket into exactly one label.",
        context="Label set with definitions. Return unknown if evidence is insufficient.",
        constraints=["Return at most 5 bullets."],
        output_contract="strict JSON")
except Exception:
    _clean = None
_dirty = "Please summarize and classify these tickets. Be helpful."

def _t5_codes(prompt):
    return {f.code for f in lint_prompt(prompt)}

check("T5 clean prompt has no findings", lambda: lint_prompt(_clean) == [])
check("T5 flags MISSING_CONTRACT", lambda: "MISSING_CONTRACT" in _t5_codes(_dirty))
check("T5 flags MISSING_ABSTAIN", lambda: "MISSING_ABSTAIN" in _t5_codes(_dirty))
check("T5 flags UNBOUNDED_OUTPUT", lambda: "UNBOUNDED_OUTPUT" in _t5_codes(_dirty))
check("T5 flags MIXED_INTENTS", lambda: "MIXED_INTENTS" in _t5_codes(_dirty))
check("T5 findings sorted by code",
      lambda: (lambda s: s == sorted(s))([f.code for f in lint_prompt(_dirty)]))

## Task 6: Build a few-shot prompt that cannot leak eval items

Few-shot examples must not include the tickets you plan to evaluate, or your results are contaminated. Build the injector with a leakage guard.

**Contract for `build_few_shot`:**
- Inputs: `base_prompt: str`, `examples: list[dict]` (each with keys `ticket_id`, `text`, `label`, `rationale`), and `eval_ticket_ids: set`.
- If any example `ticket_id` is in `eval_ticket_ids`, raise `ValueError` naming the offending ids. Check this before rendering anything.
- Otherwise append each example as a block that begins with a line `Example <ticket_id>:`, then the ticket text, then a single-record contract payload rendered as JSON inside a fenced ```json block.
- The examples must be detectable by `count_shots` (so the `Example ...:` line matters).

In [ ]:
def build_few_shot(base_prompt: str, examples: list[dict], eval_ticket_ids: set) -> str:
    leaks = [e["ticket_id"] for e in examples if e["ticket_id"] in eval_ticket_ids]
    if leaks:
        raise ValueError(f"example leakage: {leaks} appear in the evaluation set")
    blocks = []
    for e in examples:
        record = {"records": [{"ticket_id": e["ticket_id"], "label": e["label"],
                               "rationale": e["rationale"]}], "stats": {"count": 1}}
        rendered = orjson.dumps(record).decode()
        blocks.append(f"Example {e['ticket_id']}:\n"
                      f"Ticket: {e['text']}\n"
                      f"Output:\n```json\n{rendered}\n```")
    return base_prompt + "\n\n" + "\n\n".join(blocks)

In [ ]:
_eval_ids = {t["ticket_id"] for t in TICKETS}
_safe = [
    {"ticket_id": "EX-1", "text": "My card was charged twice after I canceled.",
     "label": "billing", "rationale": "Double charge after cancellation is a billing event here."},
    {"ticket_id": "EX-2", "text": "The app shows a blank white screen on launch.",
     "label": "bug", "rationale": "Blank screen on launch is broken behavior, a clear bug case."},
]
def _t6_fs():
    return build_few_shot("BASE PROMPT", _safe, _eval_ids)

def _t6_leaks() -> bool:
    try:
        build_few_shot("BASE", [{"ticket_id": "LK-1001", "text": "x", "label": "billing",
                                 "rationale": "a leaked eval item used as an example row"}], _eval_ids)
        return False
    except ValueError:
        return True

check("T6 injects two examples", lambda: count_shots(_t6_fs()) == ("few", 2))
check("T6 renders fenced json",
      lambda: (lambda s: "```json" in s and "EX-1" in s and "EX-2" in s)(_t6_fs()))
check("T6 leakage raises ValueError", _t6_leaks)

## Part E - Tally

Run this after your tasks. Green means every contract holds. Re-run the whole notebook any time to reset the counters.

In [ ]:
summary()

## Stretch goals (optional)

These are not graded by the tally above. Solutions live in the instructor solution notebook, not here.

**S1 - Add a `security` label.** Extend the allowed set to include `security` (privacy concerns, unauthorized access, data exposure) and make the contract accept it without breaking the existing checks.

**S2 - Repair near-miss output.** Write `repair_output(raw: str) -> str` that salvages a payload wrapped in prose or a fenced code block by returning just the first balanced JSON object, then confirm the repaired string passes `validate_output`.

**S3 - Rank the linter.** Add a severity to each finding (contract and mixed-intent are severity 1, abstain and unbounded are severity 2) and return findings sorted by severity first, then code.

**S4 - Validate the summarization contract.** Write `validate_summary_markdown(md: str) -> ValidationReport` that checks the L1 triage Markdown scaffold: the required line prefixes are present and no bullet exceeds 12 words.

### Stretch solutions (instructor notebook only)

The cells below are the reference solutions to S1 through S4. They run their own independent tally so the core `PASS=23` above is unaffected.

In [ ]:
reset_results()

# S1 - add a "security" label without breaking the base contract.
ALLOWED_LABELS_V2 = ALLOWED_LABELS | {"security"}

class RecordV2(Record):
    @field_validator("label")
    @classmethod
    def label_in_allowed_set(cls, v: str) -> str:  # overrides the parent validator
        if v not in ALLOWED_LABELS_V2:
            raise ValueError(f"label {v!r} not in allowed set {sorted(ALLOWED_LABELS_V2)}")
        return v

def _v2_ok(label: str) -> bool:
    try:
        RecordV2.model_validate({"ticket_id": "S", "label": label,
                                 "rationale": "a rationale that is comfortably long enough here"})
        return True
    except ValidationError:
        return False

def _base_ok(label: str) -> bool:
    try:
        Record.model_validate({"ticket_id": "S", "label": label,
                               "rationale": "a rationale that is comfortably long enough here"})
        return True
    except ValidationError:
        return False

check("S1 V2 accepts security", lambda: _v2_ok("security"))
check("S1 V2 still rejects nonsense", lambda: not _v2_ok("nonsense"))
check("S1 base contract unchanged (rejects security)", lambda: not _base_ok("security"))

In [ ]:
# S2 - salvage a near-miss payload wrapped in prose or a fenced code block.
_FENCE_RE = re.compile(r"```(?:json)?\s*(.*?)```", re.DOTALL)

def repair_output(raw: str) -> str:
    """Return the first balanced JSON object found in raw. Raise ValueError if none."""
    m = _FENCE_RE.search(raw)
    candidate = m.group(1) if m else raw
    start = candidate.find("{")
    if start == -1:
        raise ValueError("no JSON object found")
    depth = 0
    for i in range(start, len(candidate)):
        if candidate[i] == "{":
            depth += 1
        elif candidate[i] == "}":
            depth -= 1
            if depth == 0:
                return candidate[start:i + 1]
    raise ValueError("unbalanced JSON braces")

_messy = ('Sure! Here is the JSON you asked for:\n```json\n'
          '{"records": [], "stats": {"count": 0}}\n```\nHope that helps!')
check("S2 repair salvages fenced JSON", lambda: validate_output(repair_output(_messy)).ok)

def _s2_raises() -> bool:
    try:
        repair_output("no json object here at all"); return False
    except ValueError:
        return True
check("S2 repair raises when no object present", _s2_raises)

In [ ]:
# S3 - rank findings by severity, then code.
_SEVERITY = {"MISSING_CONTRACT": 1, "MIXED_INTENTS": 1,
             "MISSING_ABSTAIN": 2, "UNBOUNDED_OUTPUT": 2}

def lint_prompt_ranked(prompt: str) -> list[Finding]:
    findings = lint_prompt(prompt)
    return sorted(findings, key=lambda f: (_SEVERITY.get(f.code, 9), f.code))

_ranked = lint_prompt_ranked("Please summarize and classify these tickets. Be helpful.")
check("S3 severity-1 finding sorts first",
      lambda: _SEVERITY[_ranked[0].code] == 1)
check("S3 order is (severity, code)",
      lambda: [(_SEVERITY[f.code], f.code) for f in _ranked]
              == sorted((_SEVERITY[f.code], f.code) for f in _ranked))

In [ ]:
# S4 - validate the L1 triage summarization Markdown contract.
_SUMMARY_REQUIRED = [
    "### Support Triage Summary (L1 Leads)",
    "- Top recurring issue:",
    "- Affected modules:",
    "- Urgency trend:",
    "- Immediate actions",
    "- Open questions",
]

def validate_summary_markdown(md_text: str) -> ValidationReport:
    errors: list[str] = []
    for needed in _SUMMARY_REQUIRED:
        if needed not in md_text:
            errors.append(f"missing required line prefix: {needed!r}")
    for line in md_text.splitlines():
        if line.strip().startswith("- "):
            body = line.strip()[2:]
            if len(body.split()) > 12:
                errors.append(f"bullet exceeds 12 words: {body!r}")
    return ValidationReport(ok=not errors, errors=errors)

_good_summary = (
    "### Support Triage Summary (L1 Leads)\n"
    "- Top recurring issue: repeated double charges at checkout\n"
    "- Affected modules: billing, cart\n"
    "- Urgency trend: high because revenue events are failing\n"
    "- Immediate actions (max 3): audit charges, patch cart, notify finance\n"
    "- Open questions (max 2): scope of affected orders, refund window\n")
_overlong = "### Support Triage Summary (L1 Leads)\n- Top recurring issue:" + " word" * 20
check("S4 valid summary passes", lambda: validate_summary_markdown(_good_summary).ok)
check("S4 over-long bullet fails", lambda: not validate_summary_markdown(_overlong).ok)
check("S4 missing scaffold line fails", lambda: not validate_summary_markdown("nothing here").ok)

summary()

## Appendix - wiring the contract to a real provider (read-only)

You did not need a live model for this lab, and that is the point: the contract and validator are provider-agnostic. When you do call a real model, keep the same two-step discipline.

1. Ask the provider to emit JSON. On the Anthropic Messages API this is a beta feature enabled with the header `anthropic-beta: structured-outputs-2025-11-13`; on OpenAI it is the `strict` JSON-schema mode. Confirm the current header and field names against the provider docs at build time, they move.
2. Never trust step 1. Run the returned text through `validate_output` exactly as you did here. Provider-side structuring lowers the failure rate, it does not remove the need for client-side validation.

Note for engineers coming from other SDKs: the OpenAI `response_format` parameter has never existed on the Anthropic Messages API. Do not port it across.

Keep the model id in a config variable rather than hard-coding it inline, and load any key from an environment variable, never from the notebook.